<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/Lab1aml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Preparation and Data Loading

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os

# set device to GPU (cuda) if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# define image transformations: convert images to Tensors and normalize them using MNIST-specific mean and std deviation
transform = transforms.Compose([
    transforms.ToTensor(), # convert PIL to PyTorch Tensor
    transforms.Normalize((0.1307,),(0.3081,)) # normalize pixels for better training stability
])

# load dataset
# download and initialize the MNIST training and testing sets
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# create dataloaders: define iterators that provide data in batches of 64 and shuffle the training data
train_dataloader = DataLoader(train_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")





Using device: cpu


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 483kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.47MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.81MB/s]

Training samples: 60000
Test samples: 10000


## Define the Neural Network (5 layers)

In [3]:
class FiveLayerNN(nn.Module): # inherit from nn.Modue to create a custom model
  def __init__(self):
    super(FiveLayerNN, self).__init__() # initalize the parent class
    self.flatten = nn.Flatten() # layer to flatten the 28x28 into a 784 vector

    # layer 1: input is 784 (28x28 pixels) --> 512 hidden neurons
    self.fc1 = nn.Linear(28*28, 512)

    # layer 2: 512 input neurons --> 256 hidden neurons
    self.fc2 = nn.Linear(512, 256)

    # layer 3: 256 input neurons --> 128 hidden neurons
    self.fc3 = nn.Linear(256, 128)

    # layer 4: 128 input neurons --> 64 hidden neurons
    self.fc4 = nn.Linear(128, 64)

    # layer 5: 64 input neurons --> 10 output classes (digits 0-9)
    self.fc5 = nn.Linear(64, 10)

  def forward(self, x):
    x = self.flatten(x) # transform image to 1D vector

    # apply ReLu activation to the layers
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.relu(self.fc3(x))
    x = F.relu(self.fc4(x))

    x = self.fc5(x) # return raw scores (logits) from the final output layer
    return x

# instantiate the model and move it to the selected device
model = FiveLayerNN().to(device)
print(model)

FiveLayerNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=64, bias=True)
  (fc5): Linear(in_features=64, out_features=10, bias=True)
)


## Training and Testing Loops

In [4]:
# set hyperparameters
criterion = nn.CrossEntropyLoss() # loss function for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001) # optimizer to update weights
epochs = 5 # number of times to loop over the entire dataset

# training function
def train_loop(model, device, train_loader, optimizer, criterion, epoch):
  model.train() # set the model to training mode
  for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device) # move data to device

    optimizer.zero_grad() # clear gradients from the previous step
    output = model(data) # perform the forward pass to get prediction
    loss = criterion(output, target) # calculate the difference between prediction and truth
    loss.backward() # perform backpropagation to calculate gradients
    optimizer.step() # update the model weights based on gradients

    if batch_idx % 200 == 0: # print progress every 200 batches
      print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}')

# testing function
def test_loop(model, device, test_loader,criterion):
  model.eval() # set the model to evaluation mode
  test_loss = 0 # counter for total loss
  correct = 0 # counter for correct predictions
  with torch.no_grad(): # diable gradient calculation to save speed
    for data, target in test_loader:
      data, target = data.to(device), target.to(device) # move data to device
      output = model(data) # get predictions
      test_loss += criterion(output, target).item() # sum up the batch loss
      pred = output.argmax(dim=1, keepdim=True) # find the index with highest score
      correct += pred.eq(target.view_as(pred)).sum().item() # count correct guesses

  test_loss /= len(test_loader) # average the loss over all batches

  # print accuracy and loss for the test set
  accuracy = 100. * correct / len(test_loader.dataset)
  print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n')

# execute the training process
for epoch in range(1, epochs + 1): # loop through the dataset multiple times (5 epochs)
  print(f"--- Epoch {epoch} ---")

  # run the training loop to update model weights using the training data
  train_loop(model, device, train_dataloader, optimizer, criterion, epoch)

  # run the test loop to evaluate the model's accuracy on unseen data
  test_loop(model, device, test_dataloader, criterion)

print("Training Complete")


--- Epoch 1 ---
Train Epoch: 1 [0/60000] Loss: 2.306641
Train Epoch: 1 [12800/60000] Loss: 0.338159
Train Epoch: 1 [25600/60000] Loss: 0.178046
Train Epoch: 1 [38400/60000] Loss: 0.132973
Train Epoch: 1 [51200/60000] Loss: 0.124670

Test set: Average loss: 0.1085, Accuracy: 9659/10000 (96.59%)

--- Epoch 2 ---
Train Epoch: 2 [0/60000] Loss: 0.163120
Train Epoch: 2 [12800/60000] Loss: 0.027693
Train Epoch: 2 [25600/60000] Loss: 0.185344
Train Epoch: 2 [38400/60000] Loss: 0.175714
Train Epoch: 2 [51200/60000] Loss: 0.155740

Test set: Average loss: 0.0834, Accuracy: 9736/10000 (97.36%)

--- Epoch 3 ---
Train Epoch: 3 [0/60000] Loss: 0.043084
Train Epoch: 3 [12800/60000] Loss: 0.122888
Train Epoch: 3 [25600/60000] Loss: 0.025316
Train Epoch: 3 [38400/60000] Loss: 0.135199
Train Epoch: 3 [51200/60000] Loss: 0.124506

Test set: Average loss: 0.0717, Accuracy: 9781/10000 (97.81%)

--- Epoch 4 ---
Train Epoch: 4 [0/60000] Loss: 0.027796
Train Epoch: 4 [12800/60000] Loss: 0.021235
Train Epoch: